# Morphometry Feature Extraction

Extracts **9 classical geometric shape descriptors** from automated segmentation masks
(predicted by RefineNet) for every image in the SLiMIA dataset.

Output: `shape_features_with_metadata.csv` — required by `ImageShapeFusionTransformer`
and the ablation study.

| Feature | Description |
|---------|-------------|
| `area` | Number of pixels in the segmented spheroid |
| `perimeter` | Boundary length (pixels) |
| `eccentricity` | Ratio of focal distance to major axis (0=circle, 1=line) |
| `solidity` | Area / convex hull area — measures compactness |
| `extent` | Area / bounding box area |
| `equivalent_diameter` | Diameter of circle with same area |
| `major_axis_length` | Length of major ellipse axis |
| `minor_axis_length` | Length of minor ellipse axis |
| `circularity` | 4π·area / perimeter² — 1.0 for perfect circle |

> **Important:** morphometric descriptors are computed exclusively from **predicted** masks
> (not ground-truth), ensuring a fully automatic pipeline consistent with IPP training.
> Ground-truth masks are used only to train the segmentation model.

In [ ]:
# !pip install scikit-image tifffile --quiet

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from skimage import measure, morphology
from skimage.filters import threshold_otsu

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

In [ ]:
# Config
"""
Two modes of operation:

  MODE = 'predicted'
    Reads pre-generated predicted masks from PRED_MASK_DIR (produced by
    RefineNet inference in 02_segmentation_training.ipynb).
    This is the correct pipeline for IPP training — no GT leakage.

  MODE = 'otsu'
    Falls back to Otsu thresholding directly on the raw image if no
    predicted mask is available. Useful for quick prototyping.
"""
CSV_PATH      = "/kaggle/input/slimia-metadata/slimia_metadata.csv"
PRED_MASK_DIR = "../data/predicted_masks/"   # RefineNet output directory
OUTPUT_CSV    = "../data/shape_features_with_metadata.csv"
OUTPUT_DIR    = "./morphometry_output/"
MODE          = "predicted"   # 'predicted' | 'otsu'
MIN_AREA_PX   = 50            # discard masks smaller than this (artefacts)

os.makedirs(OUTPUT_DIR, exist_ok=True)

SHAPE_FEATURES = [
    "area", "perimeter", "eccentricity", "solidity", "extent",
    "equivalent_diameter", "major_axis_length", "minor_axis_length", "circularity"
]

print(f"Mode: {MODE}")
print(f"Output: {OUTPUT_CSV}")

## Feature Extraction Functions

In [ ]:
def load_tiff_gray(path: str) -> np.ndarray:
    """Load a TIFF as a 2D float32 array normalised to [0, 1]."""
    arr = tifffile.imread(path).astype(np.float32)
    if arr.ndim == 3:
        if arr.shape[0] in [1, 3] and arr.shape[0] < arr.shape[-1]:
            arr = np.transpose(arr, (1, 2, 0))
        arr = arr.mean(axis=-1)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
    return arr


def load_binary_mask(path: str) -> np.ndarray:
    """Load a pre-saved binary mask (uint8, values 0/1 or 0/255)."""
    arr = tifffile.imread(path)
    if arr.ndim == 3:
        arr = arr[..., 0] if arr.shape[-1] in [1, 3] else arr.mean(-1)
    mask = (arr > 0).astype(np.uint8)
    return mask


def otsu_mask(img_arr: np.ndarray, min_area: int = MIN_AREA_PX) -> np.ndarray:
    """
    Fallback: Otsu thresholding + keep largest connected component.
    Returns binary mask.
    """
    try:
        thresh = threshold_otsu(img_arr)
        binary = img_arr > thresh
    except Exception:
        binary = img_arr > 0.5

    # Remove small noise
    binary = morphology.remove_small_objects(binary, min_size=min_area)
    # Keep only largest object
    labeled = measure.label(binary)
    if labeled.max() == 0:
        return binary.astype(np.uint8)
    largest = np.argmax(np.bincount(labeled.flat)[1:]) + 1
    return (labeled == largest).astype(np.uint8)


def extract_shape_features(mask: np.ndarray) -> dict:
    """
    Extract 9 morphometric descriptors from a binary mask.
    Returns NaN for all features if no foreground is found.
    """
    nan_row = {f: np.nan for f in SHAPE_FEATURES}

    if mask.sum() < MIN_AREA_PX:
        return nan_row

    labeled = measure.label(mask)
    if labeled.max() == 0:
        return nan_row

    # Use the largest connected component
    props = measure.regionprops(labeled)
    region = max(props, key=lambda r: r.area)

    area          = float(region.area)
    perimeter     = float(region.perimeter) if region.perimeter > 0 else 1e-6
    eccentricity  = float(region.eccentricity)
    solidity      = float(region.solidity)
    extent        = float(region.extent)
    eq_diam       = float(region.equivalent_diameter)
    major_axis    = float(region.major_axis_length)
    minor_axis    = float(region.minor_axis_length)
    circularity   = float(4 * np.pi * area / (perimeter ** 2 + 1e-8))

    return {
        "area":               area,
        "perimeter":          perimeter,
        "eccentricity":       eccentricity,
        "solidity":           solidity,
        "extent":             extent,
        "equivalent_diameter": eq_diam,
        "major_axis_length":  major_axis,
        "minor_axis_length":  minor_axis,
        "circularity":        circularity,
    }

## Run Extraction Over Full Dataset

In [ ]:
df_meta = pd.read_csv(CSV_PATH)
df_meta["full_path"] = df_meta["full_path"].astype(str).str.strip()
print(f"Loaded {len(df_meta)} rows from metadata CSV")

# Build predicted mask path lookup (filename → pred_mask_path)
pred_mask_lookup = {}
if MODE == "predicted" and os.path.exists(PRED_MASK_DIR):
    for p in Path(PRED_MASK_DIR).glob("*_pred.tiff"):
        # strip _pred suffix to get original filename stem
        stem = p.stem.replace("_pred", "")
        pred_mask_lookup[stem] = str(p)
    print(f"Found {len(pred_mask_lookup)} predicted masks in {PRED_MASK_DIR}")
else:
    print(f"MODE='{MODE}' or directory not found — will use Otsu fallback")

records = []
n_predicted = 0
n_otsu      = 0
n_failed    = 0

for idx, row in tqdm(df_meta.iterrows(), total=len(df_meta),
                     desc="Extracting shape features"):
    img_path = row["full_path"]
    stem     = Path(img_path).stem
    record   = row.to_dict()
    record["image_path"] = img_path

    try:
        # Try predicted mask first
        if MODE == "predicted" and stem in pred_mask_lookup:
            mask = load_binary_mask(pred_mask_lookup[stem])
            n_predicted += 1
        else:
            # Fallback: Otsu on raw image
            img_arr = load_tiff_gray(img_path)
            mask    = otsu_mask(img_arr)
            n_otsu  += 1

        feats = extract_shape_features(mask)

    except Exception as e:
        feats = {f: np.nan for f in SHAPE_FEATURES}
        n_failed += 1

    record.update(feats)
    record["mask_source"] = "predicted" if stem in pred_mask_lookup else "otsu"
    records.append(record)

shape_df = pd.DataFrame(records)

print(f"\nExtraction complete:")
print(f"  From predicted masks : {n_predicted}")
print(f"  From Otsu fallback   : {n_otsu}")
print(f"  Failed               : {n_failed}")
print(f"  NaN rows (any feat)  : {shape_df[SHAPE_FEATURES].isnull().any(axis=1).sum()}")

In [ ]:
# Save the output CSV
shape_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(shape_df)} rows to {OUTPUT_CSV}")
shape_df[SHAPE_FEATURES].describe().round(3)

## Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, feat in zip(axes.flat, SHAPE_FEATURES):
    vals = shape_df[feat].dropna()
    ax.hist(vals, bins=50, color="steelblue", edgecolor="white")
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel("Value")
    ax.set_ylabel("Count")

plt.suptitle("Shape Feature Distributions (full dataset)", fontsize=13)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shape_feature_distributions.png")
plt.show()

In [ ]:
# Feature correlation heatmap
corr = shape_df[SHAPE_FEATURES].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, ax=ax, linewidths=0.5)
ax.set_title("Shape Feature Correlation Matrix")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shape_feature_correlation.png")
plt.show()

## Feature Variation by Cell Line (Top 10)

In [ ]:
top_lines = shape_df["cell_line"].value_counts().head(10).index
sub       = shape_df[shape_df["cell_line"].isin(top_lines)]

fig, axes = plt.subplots(3, 3, figsize=(16, 11))
for ax, feat in zip(axes.flat, SHAPE_FEATURES):
    order = sub.groupby("cell_line")[feat].median().sort_values().index
    sns.boxplot(data=sub, x="cell_line", y=feat, order=order, ax=ax,
                palette="tab10", showfliers=False)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=90, labelsize=7)

plt.suptitle("Shape Features by Cell Line (top 10, outliers hidden)", y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/shape_by_cell_line.png", bbox_inches="tight")
plt.show()

## Qualitative Check: Overlay Mask on Image

In [ ]:
def show_mask_overlay(img_path, mask_path=None, ax=None, size=256):
    img  = load_tiff_gray(img_path)
    img_r = np.array(Image.fromarray((img * 255).astype(np.uint8)).resize((size, size)))

    if mask_path and os.path.exists(mask_path):
        m = load_binary_mask(mask_path)
        m = np.array(Image.fromarray((m * 255).astype(np.uint8)).resize((size, size))) > 127
    else:
        m = otsu_mask(img_r.astype(np.float32) / 255)

    if ax is None:
        fig, ax = plt.subplots(figsize=(3, 3))
    ax.imshow(img_r, cmap="gray")
    ax.contour(m, colors=["lime"], linewidths=1)
    ax.axis("off")


sample_rows = df_meta.sample(min(8, len(df_meta)), random_state=42)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    stem      = Path(row["full_path"]).stem
    mask_path = pred_mask_lookup.get(stem, None)
    show_mask_overlay(row["full_path"], mask_path, ax)
    ax.set_title(f"{row['cell_line']}", fontsize=7)

plt.suptitle("Mask overlays (green contour) — 8 random samples", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/mask_overlays.png")
plt.show()